# Parkinson's Disorder Prediction — Model Training

This notebook trains a **hand-rolled SimpleNN** (pure PyTorch tensors, no `nn.Module`) for binary classification of Parkinson's disease.

**What's in this notebook:**
1. Reproducible global seed
2. Data loading & preprocessing (no leakage)
3. **Baseline models** (Majority-class & Logistic Regression)
4. **5-Fold Stratified Cross-Validation** for SimpleNN
5. Final SimpleNN trained on all training data
6. **Comprehensive evaluation metrics** for both Baseline and SimpleNN
7. Side-by-side comparison table
8. Model export

## 1  Imports & Global Seed

In [17]:
import json
import pickle
import random
import warnings

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

#  Reproducible Seed
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# Makes cuDNN deterministic (slight speed trade-off, only matters on GPU)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"All seeds set to {SEED}  —  results are now fully reproducible.")

All seeds set to 42  —  results are now fully reproducible.


## 2  Data Loading & Preprocessing

> **Fix vs. original:** The original notebook called `scaler.fit_transform(X)` (the **whole** dataset) for both train and test tensors — causing data leakage.  
> Here the scaler is **fit only on `X_train`** and then used to `transform` `X_test`.

In [18]:
df = pd.read_csv('model/parkinsons.data')

with open('model/important_features.json', 'r') as f:
    important_features = json.load(f)

y = df['status']
X = df[important_features]

print(f"Dataset shape : {X.shape}")
print(f"Features      : {important_features}")
print(f"Class balance : {y.value_counts().to_dict()}  (0=healthy, 1=Parkinson's)")

Dataset shape : (195, 7)
Features      : ['MDVP:Fo(Hz)', 'MDVP:Flo(Hz)', 'MDVP:RAP', 'spread1', 'spread2', 'D2', 'PPE']
Class balance : {1: 147, 0: 48}  (0=healthy, 1=Parkinson's)


In [19]:
# Stratified split preserves the 75/25 class ratio in both halves
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)

# Scale: fit ONLY on train, transform both  (no leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

# Save scaler for inference
with open('model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

y_train_np = y_train.to_numpy()
y_test_np  = y_test.to_numpy()

print(f"Train size : {X_train.shape[0]}   Test size : {X_test.shape[0]}")

Train size : 146   Test size : 49


## 3  Helpers

A single function that computes **all evaluation metrics** we care about so we can call it for every model uniformly.

In [20]:
def compute_metrics(y_true, y_pred, y_prob=None, label=""):
    """
    Returns a dict of classification metrics.
    y_prob : predicted probabilities for the positive class (needed for AUC-ROC).
             Pass None if unavailable.
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    metrics = {
        "Model"      : label,
        "Accuracy"   : round(accuracy_score(y_true, y_pred), 4),
        "Precision"  : round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall"     : round(recall_score(y_true, y_pred, zero_division=0), 4),
        "Specificity": round(specificity, 4),
        "F1"         : round(f1_score(y_true, y_pred, zero_division=0), 4),
        "MCC"        : round(matthews_corrcoef(y_true, y_pred), 4),
        "AUC-ROC"    : round(roc_auc_score(y_true, y_prob), 4) if y_prob is not None else "N/A",
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
    }
    return metrics


def print_metrics(m):
    print(f"  {'Model':<14} {m['Model']}")
    print(f"  {'Accuracy':<14} {m['Accuracy']}")
    print(f"  {'Precision':<14} {m['Precision']}")
    print(f"  {'Recall':<14} {m['Recall']}")
    print(f"  {'Specificity':<14} {m['Specificity']}")
    print(f"  {'F1':<14} {m['F1']}")
    print(f"  {'MCC':<14} {m['MCC']}")
    print(f"  {'AUC-ROC':<14} {m['AUC-ROC']}")
    print(f"  Confusion matrix  TP={m['TP']}  TN={m['TN']}  FP={m['FP']}  FN={m['FN']}")

## 4  Baseline Models

Two baselines give us a **floor** to beat:

| Baseline | Strategy |
|---|---|
| **Majority-Class** | Always predicts the most frequent label (1 — Parkinson's). Trivially high recall but zero ability to detect healthy patients. |
| **Logistic Regression** | A standard linear classifier — fast, interpretable, well-calibrated. The practical "minimum bar" for any neural network. |

In [21]:
# Majority-class baseline 
majority_class = int(np.bincount(y_train_np).argmax())
y_pred_majority = np.full_like(y_test_np, fill_value=majority_class)
# Prob = class frequency for AUC
majority_prob = (y_train_np == 1).mean()
y_prob_majority = np.full(len(y_test_np), fill_value=majority_prob)

metrics_majority = compute_metrics(
    y_test_np, y_pred_majority, y_prob_majority, label="Majority-Class"
)

print("-" * 50)
print("BASELINE — Majority-Class Classifier")
print("-" * 50)
print_metrics(metrics_majority)

--------------------------------------------------
BASELINE — Majority-Class Classifier
--------------------------------------------------
  Model          Majority-Class
  Accuracy       0.7551
  Precision      0.7551
  Recall         1.0
  Specificity    0.0
  F1             0.8605
  MCC            0.0
  AUC-ROC        0.5
  Confusion matrix  TP=37  TN=0  FP=12  FN=0


In [22]:
# Logistic Regression baseline 
lr_clf = LogisticRegression(
    class_weight='balanced',   # mirrors our weighted loss in SimpleNN
    max_iter=1000,
    random_state=SEED
)
lr_clf.fit(X_train, y_train_np)

y_pred_lr = lr_clf.predict(X_test)
y_prob_lr = lr_clf.predict_proba(X_test)[:, 1]

metrics_lr = compute_metrics(
    y_test_np, y_pred_lr, y_prob_lr, label="Logistic Regression"
)

print("-" * 50)
print("BASELINE — Logistic Regression")
print("-" * 50)
print_metrics(metrics_lr)

--------------------------------------------------
BASELINE — Logistic Regression
--------------------------------------------------
  Model          Logistic Regression
  Accuracy       0.7551
  Precision      0.931
  Recall         0.7297
  Specificity    0.8333
  F1             0.8182
  MCC            0.4926
  AUC-ROC        0.9324
  Confusion matrix  TP=27  TN=10  FP=2  FN=10


## 5  SimpleNN Definition

Hand-rolled neural network using raw PyTorch tensors (no `nn.Module`).  
Architecture: **7 → 6 → 4 → 1** with ReLU activations and weighted BCE loss to handle class imbalance.

> **Fix — `RuntimeError: Numpy is not available`**  
> This error is caused by an **ABI mismatch**: the installed PyTorch was compiled against NumPy 1.x,  
> but NumPy 2.0.2 is present in the environment. The C-level bridge that `torch.from_numpy()` uses  
> (`_ARRAY_API`) is therefore broken at import time.  
>  
> **Solution:** replace every `torch.from_numpy(arr)` with `torch.tensor(arr.copy())`.  
> `torch.tensor()` reads the array through Python/numpy's public API (no C bridge), copies the  
> data into a fresh tensor, and works regardless of the NumPy ABI version.

In [35]:
def _to_tensor(arr, dtype=torch.float64):
    """
    Safe numpy → tensor conversion that bypasses the NumPy C-API bridge.

    torch.from_numpy() shares memory via the C _ARRAY_API, which crashes with
      RuntimeError: Numpy is not available
    when PyTorch was compiled against NumPy 1.x but NumPy 2.x is installed.

    torch.tensor() copies data through Python-level buffer protocol instead,
    so it is immune to the ABI mismatch.
    np.ascontiguousarray() ensures the array is C-contiguous before the copy.
    """
    return torch.tensor(np.ascontiguousarray(arr), dtype=dtype)


class SimpleNN:
    """Hand-rolled 3-layer binary classifier (pure tensor ops, no nn.Module)."""

    def __init__(self, input_shape, seed=SEED):
        # Pin the per-model weight initialisation so every fold starts identically
        torch.manual_seed(seed)
        self.weights1 = torch.randn(input_shape, 6,  requires_grad=True, dtype=torch.float64)
        self.bias1    = torch.zeros(6,               requires_grad=True, dtype=torch.float64)
        self.weights2 = torch.randn(6, 4,            requires_grad=True, dtype=torch.float64)
        self.bias2    = torch.zeros(4,               requires_grad=True, dtype=torch.float64)
        self.weights3 = torch.randn(4, 1,            requires_grad=True, dtype=torch.float64)
        self.bias3    = torch.zeros(1,               requires_grad=True, dtype=torch.float64)

    @property
    def params(self):
        return [self.weights1, self.bias1,
                self.weights2, self.bias2,
                self.weights3, self.bias3]

    def forward(self, X):
        z1 = torch.matmul(X, self.weights1) + self.bias1
        a1 = torch.relu(z1)
        z2 = torch.matmul(a1, self.weights2) + self.bias2
        a2 = torch.relu(z2)
        z3 = torch.matmul(a2, self.weights3) + self.bias3
        return torch.sigmoid(z3)

    def loss_func(self, y_pred, y):
        """Weighted BCE — upweights minority class (healthy, label=0)."""
        epsilon = 1e-7
        y_pred  = torch.clamp(y_pred, epsilon, 1 - epsilon)
        w0 = 1.5   # minority: healthy
        w1 = 0.4  # majority: Parkinson's
        return -(w1 * y * torch.log(y_pred) +
                 w0 * (1 - y) * torch.log(1 - y_pred)).mean()

    def predict_proba(self, X_np):
        """Returns numpy array of P(class=1) for each sample."""
        X_t = _to_tensor(X_np)                   # safe conversion
        with torch.no_grad():
            out = self.forward(X_t).squeeze()     # shape: (N,)
        # .tolist() uses Python-level extraction — no NumPy C bridge needed
        return np.array(out.tolist(), dtype=np.float64)

    def predict(self, X_np, threshold=0.5):
        return (self.predict_proba(X_np) >= threshold).astype(int)


def train_simplenn(model, X_tr, y_tr, lr=0.01, epochs=200, verbose=False):
    """Gradient-descent training loop. Returns list of epoch losses."""
    # _to_tensor() replaces torch.from_numpy() — see note in SimpleNN docstring
    X_t = _to_tensor(X_tr)
    y_t = _to_tensor(y_tr.astype(np.float64))
    losses = []

    for epoch in range(1, epochs + 1):
        y_pred = model.forward(X_t)
        loss   = model.loss_func(y_pred, y_t)
        losses.append(loss.item())

        loss.backward()

        with torch.no_grad():
            for p in model.params:
                p -= lr * p.grad
                p.grad.zero_()

        if verbose and epoch % 20 == 0:
            print(f"  Epoch {epoch:>3}/{epochs}  loss={loss.item():.6f}")

    return losses


print("SimpleNN class defined.")
print("_to_tensor() helper active — torch.from_numpy() ABI issue neutralised.")

SimpleNN class defined.
_to_tensor() helper active — torch.from_numpy() ABI issue neutralised.


## 6  5-Fold Stratified Cross-Validation (SimpleNN)

Cross-validation gives an **unbiased estimate of generalisation performance** across different train/validation splits.  
We use `StratifiedKFold` to maintain the class ratio in every fold.

In [36]:
N_FOLDS = 5
LR = 0.01
EPOCHS = 200

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_metrics = []
fold_losses  = []

print(f"Running {N_FOLDS}-fold stratified CV  (lr={LR}, epochs={EPOCHS})")
print("=" * 60)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train_np), start=1):
    X_fold_tr, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_tr, y_fold_val = y_train_np[train_idx], y_train_np[val_idx]

    # Fresh model for each fold — same seed so init weights are identical
    model_cv = SimpleNN(X_fold_tr.shape[1], seed=SEED)
    losses = train_simplenn(model_cv, X_fold_tr, y_fold_tr,
                        lr=LR, epochs=EPOCHS, verbose=False)

    y_prob_val = model_cv.predict_proba(X_fold_val)
    y_pred_val = model_cv.predict(X_fold_val)

    m = compute_metrics(y_fold_val, y_pred_val, y_prob_val,
                        label=f"SimpleNN Fold {fold}")
    fold_metrics.append(m)
    fold_losses.append(losses[-1])  # final epoch loss for this fold

    print(f"Fold {fold}:  Acc={m['Accuracy']:.4f}  F1={m['F1']:.4f}  "
          f"AUC={m['AUC-ROC']:.4f}  MCC={m['MCC']:.4f}  "
          f"Recall={m['Recall']:.4f}  Spec={m['Specificity']:.4f}")

print("-" * 60)

# Aggregate CV results
cv_summary = {
    metric: np.mean([m[metric] for m in fold_metrics])
    for metric in ["Accuracy", "Precision", "Recall", "Specificity", "F1", "MCC", "AUC-ROC"]
}
cv_std = {
    metric: np.std([m[metric] for m in fold_metrics])
    for metric in ["Accuracy", "Precision", "Recall", "Specificity", "F1", "MCC", "AUC-ROC"]
}

print("\nCV Mean ± Std across folds:")
for k in cv_summary:
    print(f"  {k:<14} {cv_summary[k]:.4f} ± {cv_std[k]:.4f}")

Running 5-fold stratified CV  (lr=0.01, epochs=200)
Fold 1:  Acc=0.5667  F1=0.6667  AUC=0.6761  MCC=0.0811  Recall=0.5909  Spec=0.5000
Fold 2:  Acc=0.4138  F1=0.5405  AUC=0.2597  MCC=-0.2224  Recall=0.4545  Spec=0.2857
Fold 3:  Acc=0.5862  F1=0.7143  AUC=0.5942  MCC=-0.0300  Recall=0.6818  Spec=0.2857
Fold 4:  Acc=0.4828  F1=0.6154  AUC=0.3701  MCC=-0.1467  Recall=0.5455  Spec=0.2857
Fold 5:  Acc=0.4483  F1=0.5556  AUC=0.3831  MCC=-0.1001  Recall=0.4545  Spec=0.4286
------------------------------------------------------------

CV Mean ± Std across folds:
  Accuracy       0.4996 ± 0.0667
  Precision      0.7203 ± 0.0346
  Recall         0.5454 ± 0.0863
  Specificity    0.3571 ± 0.0904
  F1             0.6185 ± 0.0657
  MCC            -0.0836 ± 0.1035
  AUC-ROC        0.4566 ± 0.1541


## 7  Final SimpleNN — Train on Full Training Set & Evaluate on Hold-Out Test Set

In [37]:
print(f"Training final SimpleNN on all {X_train.shape[0]} training samples …")
print(f"  Architecture : {X_train.shape[1]} → 6 → 4 → 1")
print(f"  LR={LR}  Epochs={EPOCHS}")
print()

final_model  = SimpleNN(X_train.shape[1], seed=SEED)
final_losses = train_simplenn(final_model, X_train, y_train_np,
                               lr=LR, epochs=EPOCHS, verbose=True)

print(f"\nFinal training loss : {final_losses[-1]:.6f}")

Training final SimpleNN on all 146 training samples …
  Architecture : 7 → 6 → 4 → 1
  LR=0.01  Epochs=200

  Epoch  20/200  loss=1.054860
  Epoch  40/200  loss=0.907963
  Epoch  60/200  loss=0.817454
  Epoch  80/200  loss=0.755791
  Epoch 100/200  loss=0.710909
  Epoch 120/200  loss=0.676128
  Epoch 140/200  loss=0.648090
  Epoch 160/200  loss=0.625147
  Epoch 180/200  loss=0.605469
  Epoch 200/200  loss=0.587979

Final training loss : 0.587979


In [38]:
# Evaluate on the held-out test set
y_prob_nn = final_model.predict_proba(X_test)
y_pred_nn = final_model.predict(X_test)

metrics_nn = compute_metrics(
    y_test_np, y_pred_nn, y_prob_nn, label="SimpleNN (final)"
)

print("-" * 50)
print("SimpleNN — Hold-Out Test Set Results")
print("-" * 50)
print_metrics(metrics_nn)

--------------------------------------------------
SimpleNN — Hold-Out Test Set Results
--------------------------------------------------
  Model          SimpleNN (final)
  Accuracy       0.449
  Precision      0.6786
  Recall         0.5135
  Specificity    0.25
  F1             0.5846
  MCC            -0.2055
  AUC-ROC        0.3086
  Confusion matrix  TP=19  TN=3  FP=9  FN=18


## 8  Comparison: Baseline vs SimpleNN

In [ ]:
# tidy comparison DataFrame
display_cols = ["Model", "Accuracy", "Precision", "Recall",
                "Specificity", "F1", "MCC", "AUC-ROC"]

rows = [
    {k: metrics_majority[k] for k in display_cols},
    {k: metrics_lr[k]       for k in display_cols},
    # CV mean row
    {"Model": "SimpleNN (CV mean)",
     **{k: round(cv_summary[k], 4) for k in display_cols[1:]}},
    {k: metrics_nn[k]       for k in display_cols},
]

comparison_df = pd.DataFrame(rows).set_index("Model")

# Highlight the best value in each column
styled = comparison_df.style \
    .highlight_max(axis=0, color="#c6efce") \
    .format("{:.4f}") \
    .set_caption("Model Comparison — Hold-Out Test Set  (green = best per metric)")

styled

,Accuracy,Precision,Recall,Specificity,F1,MCC,AUC-ROC
Model,,,,,,,
Majority-Class,0.7551,0.7551,1.0000,0.0000,0.8605,0.0000,0.5000
Logistic Regression,0.7551,0.9310,0.7297,0.8333,0.8182,0.4926,0.9324
SimpleNN (CV mean),0.4996,0.7203,0.5454,0.3571,0.6185,-0.0836,0.4566
SimpleNN (final),0.4490,0.6786,0.5135,0.2500,0.5846,-0.2055,0.3086


In [40]:
# Plain-text version for quick reference
print(comparison_df.to_string())

                     Accuracy  Precision  Recall  Specificity      F1     MCC  AUC-ROC
Model                                                                                 
Majority-Class         0.7551     0.7551  1.0000       0.0000  0.8605  0.0000   0.5000
Logistic Regression    0.7551     0.9310  0.7297       0.8333  0.8182  0.4926   0.9324
SimpleNN (CV mean)     0.4996     0.7203  0.5454       0.3571  0.6185 -0.0836   0.4566
SimpleNN (final)       0.4490     0.6786  0.5135       0.2500  0.5846 -0.2055   0.3086


### Interpretation

| Metric | What it tells us |
|---|---|
| **Accuracy** | Overall correct predictions — misleading on imbalanced data. |
| **Precision** | Of predicted Parkinson's, how many actually have it. |
| **Recall (Sensitivity)** | Of actual Parkinson's cases, how many were caught. |
| **Specificity** | Of actual healthy cases, how many were correctly identified as healthy. |
| **F1** | Harmonic mean of Precision & Recall — good single-number summary for imbalanced tasks. |
| **MCC** | Matthews Correlation Coefficient — robust to class imbalance, ranges [−1, 1]. |
| **AUC-ROC** | Ranking quality across all thresholds; 0.5 = random, 1.0 = perfect. |

> **Key takeaway:** The Majority-Class baseline achieves high accuracy by always predicting Parkinson's, but its MCC, Specificity, and Precision expose the deception.  
> Logistic Regression sets the practical floor. SimpleNN should exceed it on MCC and AUC-ROC to justify the added complexity.

## 9  Save Model

In [ ]:
torch.save({
    "W1"         : final_model.weights1.detach(),
    "b1"         : final_model.bias1.detach(),
    "W2"         : final_model.weights2.detach(),
    "b2"         : final_model.bias2.detach(),
    "W3"         : final_model.weights3.detach(),
    "b3"         : final_model.bias3.detach(),
    "input_size" : X_train.shape[1],
    "seed"       : SEED,
    "epochs"     : EPOCHS,
    "lr"         : LR,
}, "model/model.pth")

print("Model saved to model/model.pth")
print(f"Scaler saved to model/scaler.pkl  (fitted on {X_train.shape[0]} training samples only)")